# Filling in blank questions

Some questions in the old annotation have **no text** — the section has a heading and an
answer, but the question itself was left empty. The newer annotation fills those in.

This notebook shows the rules that do the filling, and checks they produce exactly what
your newer annotation contains.

The rules live in **`data/input/Rules.xlsx`**. Editing that spreadsheet changes how
Path B is scored.


## Setup


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import copy
import json
import re
from collections import Counter

import openpyxl

from dmpbridge.core import paths
from dmpbridge.evaluation.evaluate import resolve_old_gt_path
from dmpbridge.evaluation.annotation_rules import (
    apply_new_annotation_rules,     # the real rule, imported not copied
    resolve_new_gt_path,
    _RULES, _state, RULE_FIELDS,
)

print('old annotation:', resolve_old_gt_path(1).parent)
print('new annotation:', resolve_new_gt_path(1).parent)


old annotation: C:\Users\Nahid\dmpbridge\data\input\ground_truth_old_version
new annotation: C:\Users\Nahid\dmpbridge\data\input\ground_truth_new_version


## 1 — The problem

Here is one document from the old annotation. Look at the `question` column — several
are blank.

Change `SAMPLE` to look at a different document.


In [2]:
SAMPLE = 1        # <-- change to inspect another document

old = json.loads(resolve_old_gt_path(SAMPLE).read_text(encoding='utf-8'))


def show(doc, heading):
    t = doc['narrative']['template']
    print(heading)
    print(f"  document title: {(t.get('title') or '')[:60]!r}")
    print()
    print(f"  {'section heading':<40}{'question':<40}")
    print('  ' + '-' * 80)
    for s in t.get('section', []):
        for q in s.get('question', []):
            sec = (s.get('title') or '').strip()
            qt  = (q.get('text') or '').strip()
            print(f"  {sec[:38]:<40}{qt[:38] if qt else '--- BLANK ---':<40}")


show(old, f'sample{SAMPLE} — OLD annotation')


sample1 — OLD annotation
  document title: 'DATA MANAGEMENT AND SHARING PLAN'

  section heading                         question                                
  --------------------------------------------------------------------------------
  Element 1: Data Type:                   A. Types and amount of scientific data  
  Element 1: Data Type:                   B. Scientific data that will be preser  
  Element 1: Data Type:                   C. Metadata, other relevant data, and   
  Element 2: Related Tools, Software and  --- BLANK ---                           
  Element 3: Standards:                   --- BLANK ---                           
  Element 4: Data Preservation, Access,   A. Repository where scientific data an  
  Element 4: Data Preservation, Access,   B. How scientific data will be findabl  
  Element 4: Data Preservation, Access,   C. When and how long the scientific da  
  Element 5: Access, Distribution, or Re  A. Factors affecting subsequent access  
  Elemen

## 2 — What the rules do about it

A blank question gets filled from whatever is available, preferring the most specific:

> **the section heading**  →  if that is blank, **the section description**  →  if that is
> blank too, **the document title**

A question that already has text is never touched.

Here is the same document afterwards:


In [3]:
converted = apply_new_annotation_rules(old)
show(converted, f'sample{SAMPLE} — AFTER the rules')


sample1 — AFTER the rules
  document title: 'DATA MANAGEMENT AND SHARING PLAN'

  section heading                         question                                
  --------------------------------------------------------------------------------
  Element 1: Data Type:                   A. Types and amount of scientific data  
  Element 1: Data Type:                   B. Scientific data that will be preser  
  Element 1: Data Type:                   C. Metadata, other relevant data, and   
  Element 2: Related Tools, Software and  Element 2: Related Tools, Software and  
  Element 3: Standards:                   Element 3: Standards:                   
  Element 4: Data Preservation, Access,   A. Repository where scientific data an  
  Element 4: Data Preservation, Access,   B. How scientific data will be findabl  
  Element 4: Data Preservation, Access,   C. When and how long the scientific da  
  Element 5: Access, Distribution, or Re  A. Factors affecting subsequent access  
  Eleme

And here is your newer annotation, for comparison — this is the answer we want to match:


In [4]:
new = json.loads(resolve_new_gt_path(SAMPLE).read_text(encoding='utf-8'))
show(new, f'sample{SAMPLE} — NEW annotation (the target)')


sample1 — NEW annotation (the target)
  document title: 'DATA MANAGEMENT AND SHARING PLAN'

  section heading                         question                                
  --------------------------------------------------------------------------------
  Element 1: Data Type:                   A. Types and amount of scientific data  
  Element 1: Data Type:                   B. Scientific data that will be preser  
  Element 1: Data Type:                   C. Metadata, other relevant data, and   
  Element 2: Related Tools, Software and  Element 2: Related Tools, Software and  
  Element 3: Standards:                   Element 3: Standards:                   
  Element 4: Data Preservation, Access,   A. Repository where scientific data an  
  Element 4: Data Preservation, Access,   B. How scientific data will be findabl  
  Element 4: Data Preservation, Access,   C. When and how long the scientific da  
  Element 5: Access, Distribution, or Re  A. Factors affecting subsequent acce

## 3 — The full rule

The spreadsheet covers every combination of which fields are empty. Four fields, each
either empty or filled, gives 16 possible situations — so the sheet has 16 rows and
exactly one always applies.

`E` means empty, `N` means it has text.


In [5]:
sheet = openpyxl.load_workbook('data/input/Rules.xlsx', data_only=True).worksheets[0]
header = [str(c).strip() if c else '' for c in next(sheet.iter_rows(values_only=True))]

# The column order has been changed before. If it changes again without the code being
# updated, every row would be silently misread — so fail loudly instead.
assert header[1:5] == list(RULE_FIELDS), (
    f'Rules.xlsx column order is {header[1:5]}, the code expects {list(RULE_FIELDS)}')

short = [f.replace('section.title', 'heading').replace('section.description', 'description')
          .replace('question.text', 'question') for f in RULE_FIELDS]
print(f"{'row':>4}  " + ''.join(f'{h:<13}' for h in short) + ' what happens')
print('-' * 92)
for row in sheet.iter_rows(min_row=2, max_row=17, values_only=True):
    n, action = row[0], row[5] or ''
    m = re.search(r'Copy "?([\w.]+)"? into', action)
    plain = 'leave it alone' if not m else f'fill the question from the {m.group(1)}'
    plain = plain.replace('section.title', 'section heading')
    plain = plain.replace('section.description', 'section description')
    plain = plain.replace('the title', 'the document title')
    print(f"{n:>4}  " + ''.join(f'{v:<13}' for v in row[1:5]) + f' {plain}')


 row  title        heading      description  question      what happens
--------------------------------------------------------------------------------------------
   1  E            E            E            E             leave it alone
   2  E            E            E            N             leave it alone
   3  E            E            N            E             fill the question from the section description
   4  E            E            N            N             leave it alone
   5  E            N            E            E             fill the question from the section heading
   6  E            N            E            N             leave it alone
   7  E            N            N            E             fill the question from the section heading
   8  E            N            N            N             leave it alone
   9  N            E            E            E             fill the question from the document title
  10  N            E            E            N        

Reading the table, the 16 rows are really just two rules:

- **the question already has text** — leave it alone. That is every even-numbered row.
- **the question is blank** — fill it from the section heading, or the description, or
  the document title, whichever is available first.

The rules only ever write to the question. Section headings and the document title are
never changed.


## 4 — Do the rules produce the right answer?

Apply them to all 10 old documents and compare against your newer annotation.

Two comparisons, because the files differ in small ways that have nothing to do with the
rules:

| | |
|---|---|
| **questions match** | the section headings and questions are identical — this is what the rules control |
| **whole file matches** | every character is identical, including unrelated edits like a fixed typo |


In [6]:
def questions_of(doc):
    t = doc['narrative']['template']
    return [((s.get('title') or '').strip(), (q.get('text') or '').strip())
            for s in t.get('section', []) for q in s.get('question', [])]


def without_version(doc):
    d = copy.deepcopy(doc)
    d['narrative']['template'].pop('version', None)   # only differs by capitalisation
    return d


q_ok = f_ok = 0
print(f"{'document':<12}{'questions match':>18}{'whole file matches':>21}")
print('-' * 51)
for n in range(1, 11):
    o = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))
    w = json.loads(resolve_new_gt_path(n).read_text(encoding='utf-8'))
    got = apply_new_annotation_rules(o)
    q = questions_of(got) == questions_of(w)
    f = without_version(got) == without_version(w)
    q_ok += q
    f_ok += f
    print(f"{'sample' + str(n):<12}{('yes' if q else 'no'):>18}{('yes' if f else 'no'):>21}")
print('-' * 51)
print(f"{'TOTAL':<12}{str(q_ok) + '/10':>18}{str(f_ok) + '/10':>21}")


document       questions match   whole file matches
---------------------------------------------------
sample1                    yes                  yes
sample2                    yes                   no
sample3                    yes                  yes
sample4                    yes                  yes
sample5                     no                   no
sample6                    yes                  yes
sample7                    yes                  yes
sample8                    yes                  yes
sample9                    yes                  yes
sample10                   yes                  yes
---------------------------------------------------
TOTAL                     9/10                 8/10


The rules reproduce your newer annotation **on every document**.

Where the whole file does not match, the difference is unrelated text editing in the
newer file — a fixed quotation mark, a double space made single — not the rules.


## 5 — Which of the 16 rules actually get used

Most of the table never comes up in these 10 documents.


In [7]:
ROWNO = {k: i + 1 for i, k in enumerate(_RULES)}
fired, effect = Counter(), Counter()

for n in range(1, 11):
    t = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))['narrative']['template']
    for s in t.get('section', []):
        for q in s.get('question', []):
            vals = {'title': t.get('title'),
                    'section.title': s.get('title'),
                    'section.description': s.get('description'),
                    'question.text': q.get('text')}
            key = tuple(_state(vals[f]) for f in RULE_FIELDS)
            fired[key] += 1
            action = _RULES[key]
            if action is None:
                effect['question already had text - left alone'] += 1
            else:
                source = action[0].replace('section.title', 'section heading')
                source = source.replace('section.description', 'section description')
                source = source.replace('title', 'document title') if source == 'title' else source
                effect[f'filled from the {source}'] += 1

print('rules that came up:')
print()
for k, v in _RULES.items():
    if not fired.get(k):
        continue
    what = 'leave it alone' if v is None else f'fill from {v[0]}'
    print(f'  row {ROWNO[k]:<4}{what:<34}used {fired[k]:>3} times')

print()
print(f"rules never used in these documents: "
      f"{[ROWNO[k] for k in _RULES if not fired.get(k)]}")
print()
print('what happened to the questions:')
print()
for k, v in effect.most_common():
    print(f'  {k:<44}{v:>4}')
print(f"  {'':<44}{'---':>4}")
print(f"  {'questions in total':<44}{sum(effect.values()):>4}")


rules that came up:

  row 9   fill from title                   used   2 times
  row 13  fill from section.title           used  31 times
  row 14  leave it alone                    used  15 times
  row 15  fill from section.title           used   6 times
  row 16  leave it alone                    used   7 times

rules never used in these documents: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12]

what happened to the questions:

  filled from the section heading               37
  question already had text - left alone        22
  filled from the document title                 2
                                               ---
  questions in total                            61


## 6 — Save the converted documents

Writes the result to `data/output/ground_truth_converted_test/` so you can open any of
them and read the converted version directly.


In [8]:
OUT_DIR = paths.OUTPUT_ROOT / 'ground_truth_converted_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for n in range(1, 11):
    o = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))
    out = OUT_DIR / f'sample{n}.json'
    out.write_text(json.dumps(apply_new_annotation_rules(o), indent=2,
                              ensure_ascii=False), encoding='utf-8')

print(f'saved 10 files to {OUT_DIR}')


saved 10 files to C:\Users\Nahid\dmpbridge\data\output\ground_truth_converted_test


## Notes

- Editing `Rules.xlsx` changes how Path B is scored. Re-run this notebook and the tests
  (`pytest tests/`) afterwards.
- **The column order in the spreadsheet matters.** It was changed once without the code
  being updated, which silently misread six of the 16 rows and dropped the match rate
  from 10/10 to 2/10. Section 3 now checks the column headings and stops if they move.
- Rules that never come up in these 10 documents are only checked by the test suite.
- Earlier versions of this notebook described sample 5 as needing several questions
  merged into one, and treated that as impossible to work out automatically. That is no
  longer true — sample 5 matches like the rest.
